# Trabajo Práctico Final: Simulación de Eventos Discretos
## Estudio de la eficiencia técnica, operativa y financiera en la infraestructura de una estación de carga de vehículos eléctricos a través de la simulación de eventos discretos en CABA

**Equipo:** Carlana Rivero, Loglen, Millán, Ojeda Cabrera.

**Objetivo:** Simular, mediante la metodología evento a evento, el funcionamiento de una red de estaciones de carga de vehículos eléctricos en CABA con expansión dinámica de la infraestructura, demanda variable por franja horaria y a largo plazo, fallas y mantenimiento de cargadores, y estructura de costos dinámica. Se busca determinar los valores de las variables de control (`RC`, `TMP`, `PDCE`) que equilibren congestión, arrepentimiento, capacidad ociosa y beneficio económico.

> Las celdas marcadas **[DESACTUALIZADO — TP 4]** son copia del TP 4 y deben actualizarse al modelo del TP Final. Las marcadas **[PENDIENTE]** son esqueleto sin implementar.

In [ ]:
# [DESACTUALIZADO — TP 4] verificar la ruta de Drive del TP Final al cargar los datos.
from google.colab import drive
# Montaje de Google Drive para el acceso a los datasets históricos
drive.mount('/content/drive')

## 1. Configuración del Entorno e Importación de Librerías

Se procede con la preparación del entorno de ejecución, incluyendo la instalación de la librería `fitter` para el modelado estadístico y la importación de módulos para el procesamiento de datos, visualización y simulación estocástica.

> **[DESACTUALIZADO — TP 4]** Falta el import de `dataclasses` (estado del motor). La versión instalada de `fitter` (1.8) paraleliza `fit()` por distribución (`max_workers`; `n_jobs` quedó obsoleto); cómo se aprovecha ese paralelismo en los ajustes está en la sección 4.

In [ ]:
pip install fitter

In [ ]:
# [DESACTUALIZADO — TP 4] agregar los imports del TP Final (ver nota de arriba).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit, OptimizeWarning
from fitter import Fitter, get_distributions
import math
import random
import warnings
from typing import List
import os
# Ejecución en paralelo (ejecutar_en_paralelo, sección 4): procesos creados con fork.
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

## 2. Ajuste de la Curva de Crecimiento del Parque Automotor Eléctrico

La demanda de carga no es estacionaria en el largo plazo: el parque de vehículos eléctricos (BEV) e híbridos enchufables (PHEV) crece de forma acelerada hasta un punto de inflexión, a partir del cual se desacelera hasta saturar. En esta sección se reconstruye la evolución de ese parque en CABA, se elige la curva que proyecta su crecimiento de forma más realista, se estima su techo a partir del parque automotor de CABA y del techo que alcanzó la participación de los enchufables en las ventas de los mercados maduros y se deriva de ella el factor variable en el tiempo que multiplica los intervalos entre arribos generados por el motor.

**Fuentes.** Las series están en la carpeta `Datos/` del Drive del TP Final; la versión local las lee de su copia en `Datos/`, en la raíz del repositorio:
- `Total Patentamientos CABA.xlsx`: patentamientos trimestrales de automotores por tipo y segmento en la Ciudad de Buenos Aires (Instituto de Estadística y Censos de la Ciudad, IDECBA, cuadro AC_AU_AX03, sobre datos de ACARA).
- `Distribución Patentamientos CABA.xlsx`: distribución porcentual trimestral de esos patentamientos por tipo de motor y combustible (IDECBA, cuadro AC_AU_AX08, sobre datos de ACARA).
- `EV data by country 2026.xlsx`: participación de los autos eléctricos en las ventas por país (IEA, *Global EV Outlook 2026*).
- `Parque Automotor CABA.xlsx`: parque automotor mensual de CABA (IDECBA, cuadro TPTE_PA, sobre datos de la Dirección Nacional de los Registros Nacionales de la Propiedad del Automotor, DNRPA).
- `bachelor_Pham_Minh_Khoa_2023.pdf`, en `Bibliografia/` del repositorio: Pham (2023), referencia metodológica para el ajuste de curvas de crecimiento al parque eléctrico; no se lee desde el código.

Dos datos entran como constantes, porque provienen de informes y no de series: la proporción anual de enchufables entre los electrificados patentados en Argentina (informes de electromovilidad del SIOMAA) y el porcentaje de usuarios que carga en estaciones públicas ([IEA, *Global EV Outlook 2024*](https://iea.blob.core.windows.net/assets/a9e3544b-0b12-4e15-b407-65f5c8ce1b5f/GlobalEVOutlook2024.pdf), valor de México).

### 2.1. Serie histórica de CABA

Stock trimestral de autos enchufables que cargan en estaciones públicas, reconstruido a partir de los patentamientos del primer trimestre de 2020 al segundo de 2026.

In [ ]:
# Fuentes: carpeta de datos del TP Final en Drive.
PATH_CRECIMIENTO = "/content/drive/MyDrive/Colab Notebooks/TP Final Simu/Datos/"
# openpyxl avisa que descarta extensiones de formato de Excel que no soporta; no afecta los valores leídos.
warnings.filterwarnings("ignore", message="Unknown extension is not supported")
warnings.filterwarnings("ignore", message="Print area cannot be set")

# Cuadros del IDECBA: tres filas de título y encabezado; cada año es una fila de cabecera seguida de sus trimestres.
patentamientos = pd.read_excel(PATH_CRECIMIENTO + "Total Patentamientos CABA.xlsx", sheet_name="AC_AU_AX03",
                               header=None, skiprows=3, usecols="A:C", names=["periodo", "total", "autos"])
distribucion = pd.read_excel(PATH_CRECIMIENTO + "Distribución Patentamientos CABA.xlsx", sheet_name="AC_AU_AX08",
                             header=None, skiprows=3, usecols="A,G", names=["periodo", "pct_electrificados"])
iea = pd.read_excel(PATH_CRECIMIENTO + "EV data by country 2026.xlsx", sheet_name="GEVO_EV_2026")
# Parque automotor del IDECBA: tres filas de título y encabezado; cada año es una fila de cabecera seguida de sus meses.
parque = pd.read_excel(PATH_CRECIMIENTO + "Parque Automotor CABA.xlsx", sheet_name="TPTE_PA",
                       header=None, skiprows=3, usecols="A:B", names=["periodo", "parque"])

In [ ]:
def trimestral(cuadro):
    """Filas trimestrales de un cuadro del IDECBA, con el año de la fila de cabecera que las precede."""
    periodo = cuadro["periodo"].astype(str)
    anio = periodo.str.extract(r"^(\d{4})", expand=False).astype(float).ffill()
    trimestre = periodo.str.extract(r"^([1-4])\w*\.\s*trimestre", expand=False)     # "1er. trimestre*", "3er.trimestre"
    return (cuadro.assign(anio=anio, trimestre=trimestre).dropna(subset=["trimestre"])
            .astype({"anio": int, "trimestre": int}).reset_index(drop=True))

def anual(cuadro, columna):
    """Valor de la fila de cabecera de cada año (el total anual), indexado por año; los años sin total se omiten."""
    filas = cuadro[cuadro["periodo"].astype(str).str.fullmatch(r"\d{4}(\.0)?\*?")]
    return filas.set_index(filas["periodo"].astype(str).str[:4].astype(int))[columna].dropna().astype(float)

patentamientos_q, distribucion_q = trimestral(patentamientos), trimestral(distribucion)

# Controles de la lectura: los trimestres suman el total anual de autos, y el % de electrificados de cada año es el
# promedio de sus trimestres ponderado por el total de automotores del otro cuadro, lo que confirma que los dos
# cuadros quedan alineados. Entre cuadros hay diferencias de hasta 0,005 puntos (2021) por revisiones de los datos;
# un trimestre desalineado daría diferencias de puntos enteros.
autos_anuales = anual(patentamientos, "autos")
assert np.allclose(patentamientos_q.groupby("anio")["autos"].sum().loc[autos_anuales.index], autos_anuales), \
    "Los trimestres no suman el total anual de autos."
conjunto = distribucion_q.merge(patentamientos_q, on=["anio", "trimestre"], validate="one_to_one")
pct_ponderado = ((conjunto["pct_electrificados"] * conjunto["total"]).groupby(conjunto["anio"]).sum()
                 / conjunto.groupby("anio")["total"].sum())
pct_anual = anual(distribucion, "pct_electrificados")
assert np.allclose(pct_ponderado.loc[pct_anual.index], pct_anual, rtol=0, atol=0.01), \
    "El % anual de electrificados no cierra con los trimestres."

caba = (patentamientos_q[patentamientos_q["anio"] >= distribucion_q["anio"].min()]
        .merge(distribucion_q[["anio", "trimestre", "pct_electrificados"]], on=["anio", "trimestre"], how="left",
               validate="one_to_one"))
assert caba["pct_electrificados"].notna().all() and len(caba) == len(distribucion_q), "Los cuadros no cubren los mismos trimestres."
print(f"CABA: {len(caba)} trimestres, del {caba['trimestre'].iloc[0]}.º de {caba['anio'].iloc[0]} "
      f"al {caba['trimestre'].iloc[-1]}.º de {caba['anio'].iloc[-1]}. Controles de lectura: OK "
      f"({len(autos_anuales)} totales anuales de autos, {len(pct_anual)} de % de electrificados)")

**Magnitud ajustada.** Se ajusta el **stock de autos enchufables (BEV + PHEV) que cargan en estaciones públicas** de CABA, en vehículos: es la demanda potencial de carga, con la que escala el IA del motor. Se obtiene acumulando, trimestre a trimestre, los enchufables patentados que cargan en estaciones:

`autos nuevos × % de electrificados × % de enchufables entre los electrificados × % que carga en estaciones`

Los electrificados son la categoría «Ecológico» del IDECBA (eléctricos e híbridos, enchufables o no).

**Supuestos adoptados:**
1. *Escala temporal.* Cada stock se ubica al cierre de su trimestre y cada participación en las ventas, en el punto medio. Para los países (datos anuales), el stock va al cierre del año y la participación en ventas, a mitad de año.
2. *Autos.* Se toman los patentamientos de automóviles, sin comerciales livianos ni pesados. El IDECBA publica la distribución por tipo de motor para el total de automotores; se la aplica a los automóviles.
3. *Composición del mercado.* La proporción de enchufables entre los electrificados patentados en CABA es la nacional del mismo año (SIOMAA), porque no hay un dato propio de CABA. Se aplica al flujo de patentamientos de cada trimestre.
4. *Stock inicial.* El stock de enchufables anterior a 2020 se toma nulo: los informes del SIOMAA registran 0 % de enchufables entre los electrificados patentados hasta 2019.
5. *Sin bajas.* No se descuentan bajas del parque: el enchufable más antiguo de la serie tiene seis años y medio.
6. *Carga pública.* El 29 % de los usuarios carga en estaciones y el resto en su domicilio (valor de México en IEA, *Global EV Outlook 2024*), en proporción constante en el tiempo.

In [ ]:
# Proporción de enchufables (BEV + PHEV) entre los electrificados patentados en Argentina, en %, por año.
# Fuente: informes de electromovilidad del SIOMAA (2026: primer semestre).
PCT_ENCHUFABLES_ANIO = {2020: 1.0, 2021: 1.0, 2022: 2.5, 2023: 3.8, 2024: 11.0, 2025: 21.0, 2026: 30.0}
# Fracción de los usuarios que carga en estaciones públicas; el resto carga en su domicilio (IEA, GEVO 2024, México).
PORC_CARGA_ESTACIONES = 0.29

caba["anio_stock"] = caba["anio"] + caba["trimestre"] / 4               # cierre del trimestre
caba["anio_ventas"] = caba["anio"] + (caba["trimestre"] - 0.5) / 4      # punto medio del trimestre
caba["pct_enchufables"] = caba["anio"].map(PCT_ENCHUFABLES_ANIO)
caba["pct_enchufables_ventas"] = caba["pct_electrificados"] * caba["pct_enchufables"] / 100   # % de los autos nuevos
caba["enchufables_nuevos"] = caba["autos"] * caba["pct_enchufables_ventas"] / 100
caba["stock"] = caba["enchufables_nuevos"].cumsum() * PORC_CARGA_ESTACIONES
assert caba["pct_enchufables"].notna().all(), "Falta la proporción de enchufables de algún año."

display(caba[["anio", "trimestre", "autos", "pct_electrificados", "pct_enchufables", "pct_enchufables_ventas",
              "stock"]].round(3))

### 2.2. Modelos de crecimiento

**Idea original: la logística.** La propuesta plantea la curva logística `P(t) = K / (1 + A·e^(−r·t))`, la forma clásica de una adopción en "S". Al proyectarla para CABA, el resultado no fue realista. La logística es simétrica respecto de su inflexión, que ocurre a la mitad del techo (`P = K/2`): frena al mismo ritmo al que aceleró. Como la serie de CABA está en plena aceleración, el ajuste le asigna la velocidad alta del tramo reciente y, por la simetría, la obliga a frenar igual de rápido. La curva resultante recorre toda la "S" en unos cinco años: para sostener ese ritmo, entre 2028 y 2031 tendría que ser enchufable más del 100 % de los autos nuevos, algo físicamente imposible, y a partir de mediados de 2034 la curva deja de sumar enchufables, como si en CABA casi no se patentaran más. Los números están en 2.4.

**Alternativas.** Se probaron tres curvas más flexibles:
- **Gompertz**, `P(t) = K·e^(−b·e^(−c·t))`: asimétrica, con la inflexión en `P = K/e` (37 % del techo). Crece rápido al principio y se aproxima lentamente a la saturación. En Pham (2023) es el mejor modelo para varias de las series nórdicas.
- **Richards** (logística generalizada): agrega a la logística un parámetro de forma. La serie de CABA no alcanza para determinarlo y el ajuste elige formas extremas, que aceleran hasta el final y frenan de golpe, con cuotas implícitas de enchufables por encima del 100 % de los patentamientos. Se descarta.
- **Bass**: modelo de difusión con coeficientes de innovación e imitación. Anclado al stock observado de CABA, el coeficiente de innovación queda prácticamente nulo y la curva coincide con la logística, así que no aporta nada distinto. Se descarta.

La comparación de 2.4 queda entre la logística y Gompertz. En las dos, `t` se mide en años desde el origen de cada serie (para CABA, desde el 1/1/2020).

In [ ]:
def modelo_logistico(t, K, A, r):
    """Logística: P(t) = K / (1 + A·e^(−r·t)). Simétrica: inflexión en t = ln(A)/r, con P = K/2."""
    return K / (1 + A * np.exp(-r * t))

def modelo_gompertz(t, K, b, c):
    """Gompertz: P(t) = K·exp(−b·e^(−c·t)). Asimétrica: inflexión en t = ln(b)/c, con P = K/e."""
    return K * np.exp(-b * np.exp(-c * t))

### 2.3. Techo del parque: analogía con los mercados maduros

La serie de CABA está en plena aceleración y todavía no muestra su inflexión, así que por sí sola no permite estimar el techo `K`. Se lo estima como el stock de enchufables al que tiende CABA cuando se haya renovado todo su parque: a largo plazo, la fracción enchufable del parque tiende a la participación de los enchufables en las ventas de autos nuevos, y de esos autos sólo el 29 % carga en estaciones públicas (supuesto 6 de 2.1):

`K = parque de automóviles de CABA × techo de la cuota × 29 %`

**Techo de la cuota.** Es el porcentaje máximo de enchufables en las ventas de autos nuevos al que llega un mercado. CABA está en 9,1 % (segundo trimestre de 2026) y todavía no lo muestra, así que se lo toma de los siete mercados maduros de la base de la IEA, que ya pasaron la inflexión de esa cuota: **Noruega, Suecia, Finlandia, Islandia, Dinamarca, Países Bajos y China**. Los países latinoamericanos de la base están en la misma etapa que CABA o detrás, así que todavía no muestran ni su inflexión ni su techo. El techo de cada país es el **máximo histórico** de su cuota, y el de CABA, la **mediana** de los siete. La mediana deja afuera el efecto de Noruega (97 %), un caso de política excepcional que desplaza la media de los siete casi seis puntos hacia arriba.

**Supuestos adoptados:**
1. *Parque de CABA.* Parque automotor de junio de 2026 (`T = 0`), dato provisorio. Son automotores radicados en CABA en condiciones de circular, el mismo criterio de radicación que los patentamientos de 2.1. Al basarse en el registro, incluye los vehículos que dejaron de circular sin darse de baja.
2. *Fracción de automóviles.* El cuadro del parque incluye todos los automotores (automóviles y comerciales livianos y pesados), y la cuota de la IEA es la de los automóviles. Como no hay un dato del parque por tipo de vehículo, la fracción de automóviles se aproxima con la de los patentamientos de CABA de toda la serie disponible (2016 al segundo trimestre de 2026). Es una aproximación, porque el parque acumula ventas de décadas.
3. *Parque constante.* El parque de automóviles se mantiene en el valor de `T = 0` en todo el horizonte. Entre 2016 y 2025 el parque automotor de CABA creció alrededor de 1,4 % por año.
4. *Sesgo de selección.* Los siete mercados maduros son los que más rápido adoptaron el auto eléctrico, con mayores ingresos, incentivos más fuertes y más oferta de modelos que Argentina. La mediana tiende a sobrestimar el techo de la cuota de CABA, y no hay mercados maduros menos favorables para compensarlo.

In [ ]:
PAISES_MADUROS = ["Noruega", "Suecia", "Finlandia", "Islandia", "Dinamarca", "Países Bajos", "China"]
NOMBRES_PAISES = {"Norway": "Noruega", "Sweden": "Suecia", "Finland": "Finlandia", "Iceland": "Islandia",
                  "Denmark": "Dinamarca", "Netherlands": "Países Bajos", "China": "China"}
MAGNITUDES_IEA = {"EV sales share": "pct_ventas"}

# IEA: autos (powertrain "EV" = BEV + PHEV), datos históricos; cuota de los enchufables en las ventas, en %.
paises = (iea[(iea["category"] == "Historical") & (iea["mode"] == "Cars") & (iea["powertrain"] == "EV")
              & iea["parameter"].isin(list(MAGNITUDES_IEA)) & iea["region_country"].isin(list(NOMBRES_PAISES))]
          .assign(pais=lambda d: d["region_country"].map(NOMBRES_PAISES),
                  magnitud=lambda d: d["parameter"].map(MAGNITUDES_IEA))
          .pivot_table(index=["pais", "year"], columns="magnitud", values="value")
          .reset_index().rename(columns={"year": "anio"}))

T_INICIO = 6.5                                                    # T = 0 del motor: 1/7/2026, en años desde el 1/1/2020
P_INICIO = caba["stock"].iloc[-1]                                 # stock observado en T = 0
PCT_ENCHUFABLES_CABA = caba["pct_enchufables_ventas"].iloc[-1]    # cuota en las ventas del 2.º trimestre de 2026

# Paleta categórica en orden fijo; CABA se destaca en negro en todos los gráficos.
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
COLOR_CABA = "#0b0b0b"

def formato_ejes(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(color="#e5e5e5", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend(frameon=False, fontsize=9)

fig, ax = plt.subplots(figsize=(10, 5))
for i, pais in enumerate(PAISES_MADUROS):
    datos = paises[(paises["pais"] == pais) & (paises["anio"] >= 2010)]
    ax.plot(datos["anio"] + 0.5, datos["pct_ventas"], marker="o", markersize=3, color=PALETA[i], label=pais)
ax.plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], color=COLOR_CABA, linewidth=2.2, label="CABA (trimestral)")
ax.axhline(PCT_ENCHUFABLES_CABA, color=COLOR_CABA, linewidth=1, linestyle=":",
           label=f"Cuota actual de CABA ({PCT_ENCHUFABLES_CABA:.1f} %)")
formato_ejes(ax, "Enchufables en las ventas de autos nuevos: CABA y mercados maduros", "Año", "% de las ventas")
plt.tight_layout()
plt.show()

In [ ]:
# Parque automotor de CABA en T = 0: dato mensual (fin de mes) de junio de 2026, provisorio (supuesto 1).
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre",
         "Noviembre", "Diciembre"]
periodo_parque = parque["periodo"].astype(str).str.strip().str.rstrip("*")
parque_mensual = (parque.assign(anio=periodo_parque.str.extract(r"^(\d{4})$", expand=False).astype(float).ffill(),
                                mes=periodo_parque.map({m: i + 1 for i, m in enumerate(MESES)}))
                  .dropna(subset=["mes", "parque"]))
fila_parque = parque_mensual[(parque_mensual["anio"] == 2026) & (parque_mensual["mes"] == 6)]
assert len(fila_parque) == 1, "No se encontró un único dato del parque de junio de 2026."
PARQUE_CABA = float(fila_parque["parque"].iloc[0])

# Fracción de automóviles: la de los patentamientos de CABA de toda la serie disponible (supuesto 2).
FRACCION_AUTOS = patentamientos_q["autos"].sum() / patentamientos_q["total"].sum()
fraccion_anual = patentamientos_q.groupby("anio")["autos"].sum() / patentamientos_q.groupby("anio")["total"].sum()
assert 0 < fraccion_anual.min() and fraccion_anual.max() < 1, "La fracción de automóviles quedó fuera de (0, 1)."
PARQUE_AUTOS_CABA = PARQUE_CABA * FRACCION_AUTOS

# Techo de la cuota de cada país: máximo histórico de la participación de los enchufables en las ventas.
cuotas = paises.dropna(subset=["pct_ventas"]).sort_values(["pais", "anio"])
techos = pd.DataFrame({
    "máximo histórico (%)": cuotas.groupby("pais")["pct_ventas"].max(),
    "año del máximo": cuotas.loc[cuotas.groupby("pais")["pct_ventas"].idxmax()].set_index("pais")["anio"],
    "último dato (%)": cuotas.groupby("pais")["pct_ventas"].last(),
    "año del último dato": cuotas.groupby("pais")["anio"].last(),
}).loc[PAISES_MADUROS]
techos.loc["Mediana", ["máximo histórico (%)", "último dato (%)"]] = techos[["máximo histórico (%)", "último dato (%)"]].median()
display(techos.astype({"año del máximo": "Int64", "año del último dato": "Int64"}))

S_MAX_MADUROS = techos.loc["Mediana", "máximo histórico (%)"]
K_CABA = PARQUE_AUTOS_CABA * S_MAX_MADUROS / 100 * PORC_CARGA_ESTACIONES
assert P_INICIO < K_CABA < PARQUE_AUTOS_CABA, "El techo no queda entre el stock de T = 0 y el parque de automóviles."
print(f"Parque automotor de CABA (junio de 2026): {PARQUE_CABA:,.0f} | fracción de automóviles: {100 * FRACCION_AUTOS:.1f} % "
      f"(por año, de {100 * fraccion_anual.min():.1f} % a {100 * fraccion_anual.max():.1f} %) | "
      f"parque de automóviles: {PARQUE_AUTOS_CABA:,.0f}")
print(f"Techo de la cuota (mediana de los máximos históricos): {S_MAX_MADUROS:.1f} %")
print(f"Techo de CABA: K = {PARQUE_AUTOS_CABA:,.0f} × {S_MAX_MADUROS:.1f} % × {100 * PORC_CARGA_ESTACIONES:.0f} % = "
      f"{K_CABA:,.0f} vehículos ({K_CABA / P_INICIO:.1f} veces el stock de T = 0)")

- El techo de la cuota va de 53 % (China) a 97 % (Noruega), con mediana 61 % (Suecia). En seis de los siete países el máximo histórico es el último dato (2025); la excepción es Islandia (71 % en 2023 y 62 % en 2025). La mediana es la misma con los dos criterios.
- El parque automotor de CABA en junio de 2026 es de 1 679 722 vehículos. Los automóviles son el 73,3 % de los patentamientos de 2016 a 2026 (entre 64,6 % y 78,0 % según el año), lo que da un parque de 1 230 424 automóviles.
- El techo de CABA es `K` = 1 230 424 × 61 % × 29 % = 217 662 vehículos, 129 veces el stock de `T = 0`.

### 2.4. Comparación de la logística y Gompertz

Las dos curvas usan el techo `K` de 2.3 y se anclan al stock observado en `T = 0`, lo que fija su parámetro de posición (`A` en la logística, `b` en Gompertz). Queda un solo parámetro libre, la velocidad (`r` en la logística, `c` en Gompertz), que se ajusta por mínimos cuadrados en niveles sobre la serie de CABA, **acotada a la velocidad máxima compatible con el techo de la cuota**: la pendiente máxima de la curva (`K·r/4` en la logística, `K·c/e` en Gompertz) no puede superar los enchufables que suman las ventas actuales con la cuota en su techo. Como referencia, se informan también las curvas ajustadas sin la cota.

En el tramo observado las dos curvas aceleran; se diferencian en cómo frenan, que es lo que importa para proyectar. Por eso se las compara por lo que proyectan en una ventana de 10 años desde `T = 0`, que debe cubrir el horizonte `TF` de la simulación:
- **Crecimiento** del stock a 1, 3, 4, 5 y 10 años: es la inversa del factor que multiplicará al IA. Se informan también la inflexión y el año en que la curva llega al 95 % del techo.
- **Cuota implícita**: el % de los autos nuevos que tendría que ser enchufable para producir la pendiente de la curva, `dP/dt / (ventas anuales × 29 %)`. En `T = 0` tiene que parecerse al 9,1 % observado. Nunca puede superar el 100 % (**cota física**: el stock no puede crecer más rápido que si todos los autos nuevos fueran enchufables), y no debería superar el techo de la cuota de los mercados maduros (2.3). Con la velocidad acotada, el techo se cumple por construcción con el mercado actual, así que el control queda en la cuota de `T = 0` y en el mercado de 2023–2024.
- **Renovación del parque**: el parque de automóviles de 2.3 dividido por las ventas anuales de autos nuevos, los años que tarda en renovarse al ritmo actual de patentamientos. Se lo contrasta con el año en que la curva llega al 95 % del techo.

Las ventas anuales de autos nuevos se mantienen en el nivel de los últimos cuatro trimestres. Como ese supuesto no entra en la curva, la cuota máxima se informa también con el nivel de 2023–2024 (unos 55 000 patentamientos por año).

In [ ]:
HORIZONTE_ANIOS = 10    # ventana de proyección: supuesto, debe cubrir el horizonte TF de la simulación
VENTAS_ANUALES_CABA = caba["autos"].iloc[-4:].sum()                 # últimos cuatro trimestres
COTA_CRECIMIENTO = VENTAS_ANUALES_CABA * PORC_CARGA_ESTACIONES     # vehículos/año con el 100 % de los autos nuevos enchufables
COTA_CRECIMIENTO_23_24 = caba.loc[caba["anio"].isin([2023, 2024]), "autos"].sum() / 2 * PORC_CARGA_ESTACIONES
ANIOS_RENOVACION = PARQUE_AUTOS_CABA / VENTAS_ANUALES_CABA            # años de patentamientos que suman el parque de automóviles

def logistica_anclada(t, K, r):
    """Logística que pasa por P_INICIO en T_INICIO: A queda determinado por K y r."""
    return modelo_logistico(t, K, (K / P_INICIO - 1) * np.exp(r * T_INICIO), r)

def gompertz_anclada(t, K, c):
    """Gompertz que pasa por P_INICIO en T_INICIO: b queda determinado por K y c."""
    return modelo_gompertz(t, K, np.log(K / P_INICIO) * np.exp(c * T_INICIO), c)

CURVAS = {  # nombre: (curva anclada, arranques de la velocidad, pendiente máxima / (K × velocidad))
    "Logística": (logistica_anclada, [0.3, 1.0, 2.0], 1 / 4),        # en la inflexión, P = K/2
    "Gompertz": (gompertz_anclada, [0.1, 0.3, 1.0], 1 / np.e),        # en la inflexión, P = K/e
}
COTA_VELOCIDAD = 10   # 1/año
# Velocidad máxima compatible con el techo de la cuota: la pendiente máxima de la curva no supera los enchufables que
# suman las ventas actuales con la cuota en su techo (S_MAX_MADUROS % de las ventas × 29 %).
VELOCIDAD_MAX = {curva: S_MAX_MADUROS / 100 * COTA_CRECIMIENTO / (pendiente * K_CABA)
                 for curva, (_, _, pendiente) in CURVAS.items()}
t_caba, y_caba = (caba["anio_stock"] - 2020).to_numpy(), caba["stock"].to_numpy()

def ajustar_velocidad(curva, K, cota):
    """Velocidad por mínimos cuadrados en niveles, con K fijo, acotada a (0, cota] y con varios arranques;
    devuelve la de menor error."""
    funcion, arranques, _ = CURVAS[curva]
    mejor = None
    for v0 in (min(v, 0.99 * cota) for v in arranques):
        with warnings.catch_warnings(), np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            warnings.simplefilter("ignore", OptimizeWarning)
            try:
                (v,), _ = curve_fit(lambda t, v: funcion(t, K, v), t_caba, y_caba, p0=[v0],
                                    bounds=([0], [cota]), maxfev=5000)
            except RuntimeError:      # el arranque no convergió: se prueba el siguiente
                continue
            sse = np.sum((y_caba - funcion(t_caba, K, v)) ** 2)
        if np.isfinite(sse) and (mejor is None or sse < mejor[1]):
            mejor = (v, sse)
    if mejor is None:
        raise RuntimeError(f"{curva}: ningún arranque convergió.")
    return mejor[0]

T_PROY = np.linspace(T_INICIO, T_INICIO + HORIZONTE_ANIOS, 1201)
T_LARGO = np.linspace(0, 60, 6001)      # inflexión y 95 % de K, aunque caigan fuera de la ventana
ANIOS_PROY = (1, 3, 4, 5, 10)
COLUMNAS_CRECIMIENTO = [f"× {a} año{'s' if a > 1 else ''}" for a in ANIOS_PROY]

# Cada curva se ajusta dos veces: sin acotar (sólo como referencia) y acotada a VELOCIDAD_MAX (la que se usa).
filas, curvas, velocidades, velocidades_libres = [], {}, {}, {}
for curva, (funcion, _, _) in CURVAS.items():
    for acotada in (False, True):
        v = ajustar_velocidad(curva, K_CABA, VELOCIDAD_MAX[curva] if acotada else COTA_VELOCIDAD)
        assert not (np.isclose(v, 0) or np.isclose(v, COTA_VELOCIDAD)), f"{curva}: la velocidad quedó en la cota."
        P = lambda t, f=funcion, v=v: f(t, K_CABA, v)
        if acotada:
            curvas[curva], velocidades[curva] = P, v
        else:
            velocidades_libres[curva] = v
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            p_proy, p_largo = P(T_PROY), P(T_LARGO)
        pendiente = np.gradient(p_proy, T_PROY)
        llega_95 = p_largo >= 0.95 * K_CABA
        if acotada:
            assert pendiente.max() <= S_MAX_MADUROS / 100 * COTA_CRECIMIENTO * (1 + 1e-9), \
                f"{curva}: la cuota implícita supera el techo de la cuota."
        filas.append({
            "curva": curva if acotada else f"{curva} (sin acotar)",
            **{columna: P(T_INICIO + a) / P_INICIO for columna, a in zip(COLUMNAS_CRECIMIENTO, ANIOS_PROY)},
            "inflexión (año)": 2020 + T_LARGO[np.argmax(np.gradient(p_largo, T_LARGO))],
            "95 % de K (año)": 2020 + T_LARGO[np.argmax(llega_95)] if llega_95.any() else np.nan,
            "cuota en T=0": 100 * pendiente[0] / COTA_CRECIMIENTO,
            "cuota máx.": 100 * pendiente.max() / COTA_CRECIMIENTO,
            "cuota máx. (mercado 2023–2024)": 100 * pendiente.max() / COTA_CRECIMIENTO_23_24,
            "cuota a 10 años": 100 * pendiente[-1] / COTA_CRECIMIENTO,
        })

proyeccion = pd.DataFrame(filas).set_index("curva")
print(f"K = {K_CABA:,.0f} vehículos | velocidad ajustada (acotada): logística r = {velocidades['Logística']:.3f}/año, "
      f"Gompertz c = {velocidades['Gompertz']:.3f}/año | sin acotar: r = {velocidades_libres['Logística']:.3f}/año, "
      f"c = {velocidades_libres['Gompertz']:.3f}/año")
print("Proyección desde T = 0 (stock relativo al de T = 0) y controles. Cuotas: % de enchufables en los patentamientos "
      f"implícito en la curva; observado en T = 0: {PCT_ENCHUFABLES_CABA:.1f} %; techo de la cuota en los mercados "
      f"maduros: {S_MAX_MADUROS:.1f} %.")
display(proyeccion.round(1))
print(f"Renovación del parque: {PARQUE_AUTOS_CABA:,.0f} automóviles / {VENTAS_ANUALES_CABA:,.0f} patentamientos por año "
      f"= {ANIOS_RENOVACION:.1f} años; al ritmo actual, el parque termina de renovarse hacia "
      f"{2020 + T_INICIO + ANIOS_RENOVACION:.0f}.")

In [ ]:
# Stock proyectado y cuota implícita de las dos curvas.
COLOR_CURVA = {curva: PALETA[i] for i, curva in enumerate(CURVAS)}
t_graf = np.linspace(t_caba[0], T_INICIO + HORIZONTE_ANIOS, 500)

fig, ejes = plt.subplots(1, 2, figsize=(15, 5.5))
for curva, P in curvas.items():
    estilo = dict(color=COLOR_CURVA[curva], linewidth=1.8, label=curva)
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        ejes[0].plot(2020 + t_graf, P(t_graf), **estilo)
        ejes[1].plot(2020 + T_PROY, 100 * np.gradient(P(T_PROY), T_PROY) / COTA_CRECIMIENTO, **estilo)
ejes[0].plot(caba["anio_stock"], caba["stock"], "o", color=COLOR_CABA, markersize=3, label="CABA (observado)")
ejes[0].set_ylim(bottom=0)
formato_ejes(ejes[0], "Stock de enchufables que cargan en estaciones (CABA)", "Año", "Vehículos")
ejes[1].plot(caba["anio_ventas"], caba["pct_enchufables_ventas"], "o", color=COLOR_CABA, markersize=3,
             label="CABA (observado)")
ejes[1].axhline(100, color=COLOR_CABA, linewidth=1, linestyle=":", label="Cota física")
ejes[1].axhline(S_MAX_MADUROS, color=COLOR_CABA, linewidth=1, linestyle="-.", label="Techo de la cuota (maduros)")
formato_ejes(ejes[1], "% de enchufables en los patentamientos implícito en cada curva", "Año", "% de los patentamientos")
for ax in ejes:
    ax.axvline(2020 + T_INICIO, color="#999999", linewidth=0.8)
plt.tight_layout()
plt.show()

**Análisis comparativo.**
- **Sin acotar**, las dos curvas reproducen el ritmo actual de CABA (10,6 % y 9,3 % de enchufables en `T = 0`, contra el 9,1 % observado), pero ninguna respeta el techo de la cuota. La logística comprime toda la "S" en cinco años y exige que entre 2028 y 2031 sea enchufable más del 100 % de los autos nuevos (hasta 342 %). Gompertz llega a 91 % en 2032 (136 % con el mercado de 2023–2024). El techo de 2.3 es estructural: aun con todos los autos nuevos enchufables, el stock tardaría los 14,9 años de renovación del parque en llegar a él. La velocidad sin acotar, en cambio, toma la aceleración actual de CABA. En Gompertz, además, el cociente entre la cuota máxima y la de `T = 0` depende sólo de `K / P(T=0)` (9,8 con este techo), así que ninguna velocidad reproduce el 9,1 % actual y respeta a la vez el 61 %.
- **Logística acotada** (`r` = 0,269/año). Implica sólo 1,9 % de enchufables en `T = 0`: no reproduce el presente de CABA.
- **Gompertz acotada** (`c` = 0,183/año). Implica 6,3 % de enchufables en `T = 0` y crece ×2,3, ×7,8, ×18,3 y ×59,0 a 1, 3, 5 y 10 años. Llega al techo de la cuota en 2035 y a los 10 años todavía suma enchufables (59 % de los patentamientos). Con el mercado de 2023–2024 su cuota máxima es de 91 %, dentro de la cota física. Llega al 95 % del techo en 2051, después de los 14,9 años de renovación del parque, lo que es coherente con una cuota que arranca en 9,1 % y no en su techo.
- Sin acotar, la logística ajusta mejor el tramo observado, pero ahí las dos curvas aceleran y sólo se diferencian en cómo frenan: ajustar mejor el pasado no la favorece para proyectar.

**Conclusión.** Se adopta **Gompertz con el techo `K` de 2.3 y la velocidad acotada**: de las cuatro variantes, es la única que respeta el techo de la cuota y la cota física y se mantiene cerca del ritmo actual de CABA. Es una solución de compromiso: subestima el ritmo actual (6,3 % contra 9,1 % de enchufables en `T = 0`), lo que hace conservadora a la proyección de los primeros años (×2,3 a un año, contra ×3,2 sin acotar). Corrige la forma de la curva planteada en la propuesta.

### 2.5. Factor de crecimiento aplicado al Intervalo entre Arribos

Función que, dado el instante `T` de la simulación (minutos desde el inicio), devuelve el factor que multiplica el `IA` generado. El factor se calcula sobre el stock, `P(T=0) / P(T)`, porque los arribos a las estaciones los generan todos los autos enchufables en circulación, no sólo los que se incorporan en ese momento. Vale 1 en `T = 0` y decrece a medida que crece el parque: el IA se achica rápido durante la aceleración y cada vez más lento en la desaceleración, hasta estabilizarse en `P(T=0) / K` cuando el parque se satura.

In [ ]:
# Curva adoptada: Gompertz con el techo K de 2.3 (2.4).
# PARAMS_CRECIMIENTO = (K, b, c): K en vehículos, c en 1/año, t en años desde el 1/1/2020.
C_CRECIMIENTO = velocidades["Gompertz"]
PARAMS_CRECIMIENTO = (K_CABA, np.log(K_CABA / P_INICIO) * np.exp(C_CRECIMIENTO * T_INICIO), C_CRECIMIENTO)
MIN_POR_ANIO = 12 * 30 * 24 * 60    # 518 400 min: año de 12 meses de 30 días, la misma convención que TCE

def P_crecimiento(t, params=PARAMS_CRECIMIENTO):
    """Stock de enchufables que cargan en estaciones en CABA (Gompertz); t en años desde el 1/1/2020."""
    K, b, c = params
    return K * math.exp(-b * math.exp(-c * t))     # K·exp(−b·e^(−c·t))

def factor_crecimiento(T, params=PARAMS_CRECIMIENTO):
    """Factor que multiplica el IA generado en el instante T (min desde el inicio de la simulación).
    Es P(T = 0) / P(T): vale 1 al inicio y decrece hacia P(T = 0) / K a medida que el parque satura."""
    return P_crecimiento(T_INICIO, params) / P_crecimiento(T_INICIO + T / MIN_POR_ANIO, params)

print("Gompertz adoptada: K = {:,.0f} vehículos | b = {:.4f} | c = {:.4f}/año".format(*PARAMS_CRECIMIENTO))

# Chequeos sobre el factor (valores, no ausencia de excepciones).
T_grilla = np.linspace(0, HORIZONTE_ANIOS * MIN_POR_ANIO, 500)
factor_grilla = np.array([factor_crecimiento(T) for T in T_grilla])
limite = P_INICIO / PARAMS_CRECIMIENTO[0]
assert math.isclose(P_crecimiento(T_INICIO), P_INICIO), "La curva no pasa por el stock observado en T = 0."
assert all(math.isclose(P_crecimiento(T_INICIO + a), curvas["Gompertz"](T_INICIO + a)) for a in ANIOS_PROY), \
    "La curva adoptada no coincide con la Gompertz de 2.4."
assert math.isclose(factor_grilla[0], 1.0), "El factor no vale 1 en T = 0."
assert np.all(np.diff(factor_grilla) < 0), "El factor no es estrictamente decreciente."
assert math.isclose(factor_crecimiento(1e12), limite, rel_tol=1e-9), "El factor no tiende a P(T=0)/K."
assert factor_grilla[-1] > limite, "El factor cruzó su límite inferior."

display(pd.DataFrame([{"T (años)": a, "factor sobre el IA": factor_crecimiento(a * MIN_POR_ANIO),
                       "flujo de arribos relativo a T = 0": 1 / factor_crecimiento(a * MIN_POR_ANIO)}
                      for a in (1, 3, 5, 10)]).set_index("T (años)").round(3))

In [ ]:
# Factor sobre el IA en la ventana de proyección.
anios_grilla = 2020 + T_INICIO + T_grilla / MIN_POR_ANIO
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(anios_grilla, factor_grilla, color=PALETA[0], linewidth=1.8)
ax.axhline(limite, color=COLOR_CABA, linewidth=1, linestyle=":", label=f"Límite al saturar, P(T=0) / K = {limite:.3f}")
ax.set_yscale("log")
formato_ejes(ax, "Factor de crecimiento sobre el IA (Gompertz)", "Año", "Factor sobre el IA (escala logarítmica)")
plt.tight_layout()
plt.show()

**Factor adoptado** (Gompertz): `factor_crecimiento(T)` vale 0,444 a 1 año, 0,129 a 3, 0,054 a 5 y 0,017 a 10. Es decir, el flujo de arribos de cada estación se multiplica por 2,3, 7,8, 18,3 y 59,0 respecto de `T = 0`. La función es escalar (`math.exp`), está definida a nivel de módulo para que sea serializable en los procesos del barrido, y recibe los parámetros como argumento.

`T = 0` supone que las FDPs de IA ajustadas en la sección 5 representan la demanda de una estación al 1/7/2026. Ese supuesto se explicita también al ajustarlas.

## 3. Carga de Datos Inicial

En esta sección se cargan y depuran los dos datasets del modelo: el de sesiones de carga (3.1), del que salen la FDP de `TC` (sección 4), las seis FDPs de intervalo entre arribos (sección 5) y la regresión energía–tiempo (sección 7), y el de reportes de fallas de cargadores (3.2), del que salen las FDPs de falla y reparación (sección 6).

### 3.1. Sesiones de carga (Dataset California)

**Fuente.** `Dataset California.csv`, en la carpeta `Datos/` del Drive del TP Final; la versión local lo lee de su copia en `Datos/`, en la raíz del repositorio. Es el registro de sesiones de la red pública de carga de la ciudad de Palo Alto (California, EE. UU.) entre julio de 2011 y diciembre de 2020, el mismo dataset del TP 4. Cada fila es una sesión: estación y puerto, inicio (`Start Date`, en hora local y con resolución de un minuto), tiempo efectivo de carga (`Charging Time (hh:mm:ss)`) y energía entregada (`Energy (kWh)`).

In [ ]:
# Carpeta de datos del TP Final en Drive (la misma de la sección 2); de acá se lee también el dataset de 3.2.
PATH_DATOS = "/content/drive/MyDrive/Colab Notebooks/TP Final Simu/Datos/"

df_california = pd.read_csv(PATH_DATOS + "Dataset California.csv", low_memory=False)
print(f"Dataset California: {len(df_california):,} sesiones de {df_california['Station Name'].nunique()} estaciones.")

In [ ]:
# Inicio de la sesión en hora local de Palo Alto (PST/PDT): es la que define la franja horaria y el tipo de día.
df_california["Start Date"] = pd.to_datetime(df_california["Start Date"], format="%m/%d/%Y %H:%M")
df_california["Charging Time (minutes)"] = (pd.to_timedelta(df_california["Charging Time (hh:mm:ss)"])
                                            .dt.total_seconds() / 60)

VENTANA_DATOS = (pd.Timestamp("2018-01-01"), pd.Timestamp("2020-03-01"))   # [inicio, fin): ver el supuesto de abajo

por_mes = (df_california.set_index("Start Date").resample("MS")
           .agg(sesiones=("Station Name", "size"), estaciones=("Station Name", "nunique")))
anio_mes = por_mes.index.year + (por_mes.index.month - 1) / 12
ventana_anios = [v.year + (v.month - 1) / 12 for v in VENTANA_DATOS]

fig, ejes = plt.subplots(1, 2, figsize=(15, 4.5))
ejes[0].plot(anio_mes, por_mes["sesiones"], color=PALETA[0], linewidth=1.6, label="Sesiones por mes")
ejes[1].step(anio_mes, por_mes["estaciones"], where="post", color=PALETA[1], linewidth=1.6,
             label="Estaciones con sesiones en el mes")
for ax, titulo, ylabel in zip(ejes, ["Sesiones de carga por mes (Palo Alto)", "Estaciones activas por mes (Palo Alto)"],
                              ["Sesiones", "Estaciones"]):
    ax.axvspan(*ventana_anios, color="#999999", alpha=0.15, label="Ventana de estudio")
    formato_ejes(ax, titulo, "Año", ylabel)
plt.tight_layout()
plt.show()

en_ventana = por_mes[(por_mes.index >= VENTANA_DATOS[0]) & (por_mes.index < VENTANA_DATOS[1])]
print(f"Ventana de estudio: {len(en_ventana)} meses, entre {en_ventana['sesiones'].min():,} y "
      f"{en_ventana['sesiones'].max():,} sesiones por mes, entre {en_ventana['estaciones'].min()} y "
      f"{en_ventana['estaciones'].max()} estaciones activas.")
print(f"Abril de 2020: {por_mes.loc['2020-04-01', 'sesiones']:,} sesiones.")

**Ventana de estudio: del 1/1/2018 al 29/2/2020.** Las FDPs se ajustan sólo sobre este período, por tres razones:
- *Crecimiento de la red.* Entre 2011 y 2017 la red de Palo Alto pasa de 5 a 27 estaciones y la demanda crece con ella; a mediados de 2017 cae de golpe (de unas 4 900 a unas 3 500 sesiones por mes). El crecimiento de largo plazo ya lo modela el factor de la sección 2: ajustar los IA sobre la serie completa mezclaría la expansión de la red de Palo Alto con la variabilidad de corto plazo que tienen que representar las FDPs.
- *Pandemia.* Desde marzo de 2020 la demanda se desploma (487 sesiones en abril) y no se recupera dentro del dataset.
- *Estabilidad.* En la ventana la red tiene 33 estaciones (entre 27 y 33 con sesiones cada mes) y la demanda se mantiene entre 3 063 y 4 454 sesiones por mes, sin tendencia marcada. Son 26 meses completos, así que cada franja y cada tipo de día queda representado en todas las épocas del año.

La ventana también es la base de la regresión energía–tiempo de la sección 7, que usa el mismo `df`.

In [ ]:
# Sesiones de la ventana de estudio, sin filas repetidas.
en_ventana = df_california[(df_california["Start Date"] >= VENTANA_DATOS[0]) & (df_california["Start Date"] < VENTANA_DATOS[1])]
df = en_ventana.drop_duplicates().sort_values("Start Date", ignore_index=True)
assert df["Start Time Zone"].isin(["PST", "PDT"]).all(), "Hay sesiones de la ventana registradas fuera de la hora local."
print(f"{len(df):,} sesiones de {df['Station Name'].nunique()} estaciones, del {df['Start Date'].min():%d/%m/%Y} al "
      f"{df['Start Date'].max():%d/%m/%Y} ({len(en_ventana) - len(df)} filas repetidas descartadas).")

# Semilla de los sorteos de esta parte del notebook (desagregación de los instantes de arribo y chequeos de los
# generadores), fijada antes de ver los resultados.
SEMILLA_DATOS = 2026

### 3.2. Reportes de fallas de cargadores (Dataset Escocia)

**Fuente.** `Dataset Escocia.xlsx`, en la misma carpeta de datos (`PATH_DATOS`): registro de fallas de cargadores de la red pública de Escocia. Las hojas `2021` y `2022` tienen una fila por reporte de falla, con el cargador (`Charging point`), el instante del reporte (`Date reported`), la causa (`Root cause`, `CPS Fault` —falla en la red ChargePlace Scotland— en todas las filas) y el instante de cierre (`Closed date`), del 27/7/2021 al 29/9/2022. La hoja `2020` sólo trae totales mensuales de reportes de otro operador (BP Pulse), sin cargador ni instantes, así que no sirve para `TFC` ni `TRC` y no se usa.

**Supuestos de la depuración:**
1. *Inicio del registro.* Se descartan los reportes anteriores al 1/9/2021. El registro arranca a fines de julio de 2021 con pocos reportes (25 en julio y 222 en agosto, contra entre 549 y 921 por mes de septiembre de 2021 a agosto de 2022) y no registra ningún cierre antes del 26/8/2021, así que las reparaciones de esas semanas quedarían infladas.
2. *Reportes duplicados e inconsistentes.* Se descartan los reportes duplicados (mismo cargador y mismo instante) y los que cierran antes o en el mismo instante en que se reportan, una inconsistencia del registro.

In [ ]:
INICIO_REGISTRO_FALLAS = pd.Timestamp("2021-09-01")   # supuesto 1

hojas = pd.read_excel(PATH_DATOS + "Dataset Escocia.xlsx", sheet_name=["2021", "2022"])
reportes = (pd.concat(hojas.values(), ignore_index=True)
            .rename(columns={"Charging point": "cargador", "Date reported": "reporte", "Root cause": "causa",
                             "Closed date": "cierre"}))
reportes["cargador"] = reportes["cargador"].astype(str).str.strip()     # hay identificadores numéricos y alfanuméricos
assert (reportes["causa"] == "CPS Fault").all(), "Hay reportes con otra causa."
print(f"Registro completo: {len(reportes):,} reportes de {reportes['cargador'].nunique():,} cargadores, "
      f"del {reportes['reporte'].min():%d/%m/%Y} al {reportes['reporte'].max():%d/%m/%Y}; primer cierre: "
      f"{reportes['cierre'].min():%d/%m/%Y}.")
print("Reportes por mes:", reportes.set_index("reporte").resample("MS").size().to_dict())

fallas = reportes[reportes["reporte"] >= INICIO_REGISTRO_FALLAS]
duplicados = fallas.duplicated(["cargador", "reporte"])
fallas = fallas[~duplicados]
inconsistentes = fallas["cierre"] <= fallas["reporte"]     # NaT compara como False: los abiertos se conservan
desfase = (fallas.loc[inconsistentes, "reporte"] - fallas.loc[inconsistentes, "cierre"]).dt.total_seconds()
fallas = fallas[~inconsistentes].sort_values(["cargador", "reporte"], ignore_index=True)
print(f"Descartados: {len(reportes) - len(fallas) - duplicados.sum() - inconsistentes.sum():,} anteriores al "
      f"{INICIO_REGISTRO_FALLAS:%d/%m/%Y}, {duplicados.sum()} duplicados y {inconsistentes.sum()} con cierre no posterior "
      f"al reporte (el cierre se adelanta hasta {desfase.max():.0f} s).")
print(f"Quedan {len(fallas):,} reportes de {fallas['cargador'].nunique():,} cargadores; "
      f"{fallas['cierre'].isna().sum()} siguen abiertos al final del registro.")

## 4. Obtención de la FDP del Tiempo de Carga (TC)

El Tiempo de Carga representa la duración de la sesión de carga. Se utiliza la columna `Charging Time (hh:mm:ss)` del dataset mapeada a minutos para determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

**Procedimiento de ajuste**, el mismo para todas las FDPs del modelo (secciones 4 a 6): `Fitter().fit()` sobre todas las distribuciones continuas de `scipy.stats` → ranking por `ks_statistic`, contrastado con `sumsquare_error` y `kl_div` → elección justificada → `obtener_dist()` → generador truncado al rango observado.

- *Distribuciones.* Se ajustan todas las que `fitter` prueba por defecto (`get_distributions()`), sin preseleccionar candidatas: las distribuciones continuas univariadas de `scipy.stats`, 108 en la versión instalada. De la lista de `fitter` quedan afuera las cinco entradas que no son distribuciones univariadas (`_fit`, `rv_continuous`, `rv_histogram`, `multivariate_normal` y `vonmises_fisher`) y dos cuyo ajuste es inviable: `levy_stable` y `studentized_range` calculan la densidad por integración numérica, y una sola evaluación de la verosimilitud sobre los 99 077 tiempos de carga tarda unos 70 s y 11 min, respectivamente. Como el ajuste por máxima verosimilitud necesita del orden de mil evaluaciones, cada una tardaría de horas a días por serie, más de lo que dura una sesión de Colab. Las de soporte en toda la recta se incluyen porque el truncamiento al rango observado descarta la parte negativa. Algunas no ajustan, por limitaciones de `scipy` o porque `fitter` descarta el resultado (`irwinhall`, `kstwo`, `vonmises` y, en algunas series, `geninvgauss`): la salida de cada ajuste las informa.
- *Sin timeout.* `fitter` descarta la distribución cuyo ajuste supera su *timeout* (30 s por defecto), así que el conjunto evaluado dependería de la velocidad de la máquina y la elección no sería reproducible. Se lo desactiva (`timeout=None`): cada distribución se ajusta hasta que termina el optimizador. Las nueve series suman unos 30 min de CPU, y el ajuste más largo, el de una distribución a una serie, tarda unos 4 min.
- *Criterio.* Con miles de observaciones, el p-valor del test de Kolmogorov-Smirnov es prácticamente nulo para cualquier distribución, así que no sirve para aceptar o rechazar. Se ordena por el estadístico `ks_statistic`, la máxima distancia entre la función de distribución empírica y la ajustada, y se controla la forma con `sumsquare_error` y `kl_div`, que comparan la densidad con el histograma.
- *Truncamiento.* El generador muestrea la FDP elegida por transformada inversa, truncada al rango observado (`rvs_truncado`, del TP 4), para no generar valores fuera del dominio real. Como se simula con la FDP truncada, se controla que su media se parezca a la media observada.
- *Paralelismo.* La versión instalada de `fitter` (1.8) paraleliza `fit()` por distribución (`max_workers`; `n_jobs` quedó obsoleto). En lugar de anidar ese paralelismo dentro del de las series, cada ajuste se reparte en tareas independientes (serie × distribución) que corren en procesos separados (`ProcessPoolExecutor` en Colab, `joblib` en la versión local), con `fit(max_workers=1)` dentro de cada tarea. El reparto es dinámico: cada proceso que termina una tarea toma la siguiente, así que las distribuciones lentas no frenan a las demás. Así se ocupan todos los núcleos tanto con una sola serie (`TC`) como con varias (las seis de IA, o `TFC` y `TRC`).

In [ ]:
# Todas las distribuciones continuas que fitter prueba por defecto (ver el procedimiento de ajuste).
# Se excluyen las entradas que no son distribuciones univariadas y las dos de ajuste inviable (densidad por integración
# numérica: de horas a días por serie).
EXCLUIDAS = ["levy_stable", "studentized_range"]
DISTRIBUCIONES = [d for d in get_distributions()
                  if isinstance(getattr(stats, d), stats.rv_continuous) and d not in EXCLUIDAS]

def ajustar_parte(nombre, datos, distribucion):
    """Tarea de un proceso worker: ajusta una distribución a una serie, sin timeout. Devuelve (nombre, Fitter)."""
    f = Fitter(datos, distributions=[distribucion], timeout=None, verbose=False)
    f.fit(max_workers=1)      # sin paralelismo anidado: el reparto entre procesos lo hace ajustar_fdps
    return nombre, f

def ajustar_fdps(series):
    """Ajusta todas las distribuciones a cada serie de {nombre: datos}, repartiendo las tareas (serie × distribución)
    entre los procesos. Devuelve {nombre: Fitter} con los ajustes de cada serie reunidos."""
    tareas = [(nombre, np.asarray(datos, dtype=float), distribucion)
              for nombre, datos in series.items() for distribucion in DISTRIBUCIONES]
    resultados = ejecutar_en_paralelo(ajustar_parte, tareas)
    ajustes = {}
    for nombre, datos in series.items():
        partes = [parte for n, parte in resultados if n == nombre]
        f = Fitter(np.asarray(datos, dtype=float), distributions=DISTRIBUCIONES, verbose=False)
        for parte in partes:
            f.fitted_param.update(parte.fitted_param)
            f.fitted_pdf.update(parte.fitted_pdf)
        f.df_errors = pd.concat([parte.df_errors for parte in partes]).sort_index()
        assert sorted(f.df_errors.index) == sorted(DISTRIBUCIONES), f"{nombre}: faltan distribuciones en el ajuste."
        fallidas = f.df_errors.index[np.isinf(f.df_errors["ks_statistic"])].tolist()
        print(f"{nombre}: {len(datos):,} datos, {len(DISTRIBUCIONES) - len(fallidas)} de {len(DISTRIBUCIONES)} "
              "distribuciones ajustadas" + (f" (no ajustaron: {', '.join(fallidas)})" if fallidas else ""))
        ajustes[nombre] = f
    return ajustes

def mostrar_mejores_dist(f, nombre, criterio="ks_statistic", n=8):
    """Grafica el histograma con las cinco mejores según el criterio y muestra las n mejores con las tres métricas y el
    puesto que ocupa cada una en el ranking de todas las distribuciones."""
    plt.figure(figsize=(8, 5))
    f.summary(Nbest=5, method=criterio, clf=False)
    plt.title(f"{nombre}: las cinco mejores por {criterio}", fontsize=11)
    plt.xlabel(f"{nombre} (minutos)")
    plt.ylabel("Densidad de probabilidad")
    plt.show()
    metricas = ["ks_statistic", "sumsquare_error", "kl_div"]
    tabla = f.df_errors[metricas].copy()
    for metrica in metricas:
        tabla[f"puesto {metrica}"] = tabla[metrica].rank(method="min", na_option="bottom").astype(int)   # un NaN queda último
    display(tabla.sort_values(criterio).head(n))

In [ ]:
# Ejecución en paralelo (Colab): procesos creados con fork, que heredan las funciones definidas en el notebook.
# El trabajo es CPU-bound: procesos y no hilos, por el GIL; tantos workers como vCPUs (2 en el entorno CPU de Colab).

def ejecutar_en_paralelo(funcion, tareas):
    """Aplica funcion(*tarea) a cada tarea en procesos separados y devuelve los resultados en el orden de las tareas."""
    with ProcessPoolExecutor(max_workers=min(len(tareas), os.cpu_count()),
                             mp_context=multiprocessing.get_context("fork")) as ejecutor:
        return list(ejecutor.map(funcion, *zip(*tareas)))

print(f"Workers disponibles: {os.cpu_count()}")

In [ ]:
# Tiempo efectivo de carga de cada sesión de la ventana de estudio, en minutos.
tc = df["Charging Time (minutes)"].to_numpy()
ajustes_tc = ajustar_fdps({"TC": tc})
mostrar_mejores_dist(ajustes_tc["TC"], "TC")

**Elección de la FDP de `TC`: `dpareto_lognorm` (doble Pareto-lognormal).** Es la primera por `ks_statistic` (0,0123) y por `sumsquare_error`, y la tercera por `kl_div`, con un valor (0,0113) prácticamente igual al de las dos primeras (`mielke` y `burr`, 0,0111). Es una lognormal cuyas dos colas decaen como potencias (de tipo Pareto), más pesadas que las de la lognormal. `burr` (Burr tipo III) queda segunda (`ks_statistic` 0,0197), y la `gumbel_r` que eligió el TP 4, quinta (0,0244), sobre los datos de la ventana de estudio. La truncada al rango observado retiene el 99,96 % de la probabilidad, y su media (115,7 min) difiere en −0,5 % de la observada (116,3 min).

*Costo de generación.* La `ppf` de `dpareto_lognorm` no tiene forma cerrada y `scipy` la calcula numéricamente: cada valor generado tarda unos 4 ms, contra 0,06 ms con `burr`. Si en el motor ese costo pesa, la inversa se puede tabular una vez al inicio de la corrida.

In [ ]:
class FDPTruncada:
    """FDP ajustada y truncada al rango observado [low, high]. Se muestrea por transformada inversa uniforme entre
    cdf(low) y cdf(high), sin acumular probabilidad en los bordes (rvs_truncado del TP 4). Los límites en probabilidad
    se calculan una sola vez, para no recalcularlos en cada sorteo del motor."""

    def __init__(self, dist, low, high):
        self.dist, self.low, self.high = dist, float(low), float(high)
        self.p_low, self.p_high = float(dist.cdf(low)), float(dist.cdf(high))

    def rvs_truncado(self, rng=np.random):
        """Un valor de la FDP truncada. rng: generador de la corrida (np.random.Generator o random.Random), para que
        cada corrida tenga su propia semilla."""
        return float(self.dist.ppf(rng.uniform(self.p_low, self.p_high)))

    def media(self):
        """Esperanza de la FDP truncada a [low, high]."""
        return float(self.dist.expect(lb=self.low, ub=self.high, conditional=True))

def mostrar_fdp(title, xlabel, f, fdp, data):
    plt.figure(figsize=(7, 5))
    plt.title(title, fontsize=11)
    plt.xlabel(xlabel)
    plt.ylabel("Densidad de probabilidad")
    plt.hist(data, bins=100, density=True, alpha=0.5)
    f.plot_pdf(names=[fdp])
    plt.show()

def obtener_dist(f, funcion, nombre, data):
    """FDP truncada al rango observado, lista para el motor, a partir de un Fitter ajustado."""
    if funcion not in f.fitted_param:
        disponibles = list(f.fitted_param.keys())
        raise ValueError(f"'{funcion}' no fue ajustada. Disponibles: {disponibles}")

    params_tuple = f.fitted_param[funcion]
    dist_class   = getattr(stats, funcion)
    param_names  = (dist_class.shapes.split(", ") if dist_class.shapes else []) + ["loc", "scale"]
    params_dict  = {nombre_param: round(float(valor), 4) for nombre_param, valor in zip(param_names, params_tuple)}
    fdp = FDPTruncada(dist_class(*params_tuple), np.min(data), np.max(data))

    print(f"La función elegida para {nombre} es: {funcion}")
    print(f"Con los parámetros: {params_dict}")
    print(f"Acotada entre: {fdp.low:.3f} y {fdp.high:.3f} (probabilidad retenida: {fdp.p_high - fdp.p_low:.4f})")

    mostrar_fdp(f"FDP para {nombre}", f"{nombre} (minutos)", f, funcion, data)
    return fdp

FDP_TC = obtener_dist(ajustes_tc["TC"], "dpareto_lognorm", "TC", tc)

def generar_tc(rng=np.random):
    """Generador estocástico de Tiempo de Carga para el motor de simulación."""
    return FDP_TC.rvs_truncado(rng)

### 4.1. Tiempo de carga promedio (TCP) y factor de impaciencia (FI)

In [ ]:
def verificar_generador(fdp, generador, datos, nombre, rng, n=20_000):
    """Chequea un generador contra su FDP truncada: todos los valores dentro del rango observado y media muestral a
    menos de 4 errores estándar de la media de la FDP truncada (tolerancia fijada de antemano). Devuelve una fila de
    resumen que compara además la media de la FDP truncada con la media observada."""
    muestra = np.array([generador(rng) for _ in range(n)])
    media_fdp, media_obs = fdp.media(), float(np.mean(datos))
    cuatro_errores = 4 * muestra.std(ddof=1) / math.sqrt(n)
    assert fdp.low <= muestra.min() and muestra.max() <= fdp.high, f"{nombre}: el generador sale del rango observado."
    assert abs(muestra.mean() - media_fdp) < cuatro_errores, \
        f"{nombre}: la media generada ({muestra.mean():.3f}) no reproduce la de la FDP truncada ({media_fdp:.3f})."
    return {"serie": nombre, "media observada": media_obs, "media FDP truncada": media_fdp,
            "diferencia (%)": 100 * (media_fdp / media_obs - 1), "media generada": muestra.mean(),
            "tolerancia (4 err. est.)": cuatro_errores}

TCP = FDP_TC.media()     # E[TC] de la FDP ajustada y truncada al rango observado (propuesta)
FI = 41.7 / TCP          # 41,7 min: paciencia media de los usuarios que esperan (EAFO, ver la propuesta)

display(pd.DataFrame([verificar_generador(FDP_TC, generar_tc, tc, "TC", np.random.default_rng(SEMILLA_DATOS))])
        .set_index("serie").round(3))
print(f"TCP = {TCP:.2f} min | FI = 41,7 / TCP = {FI:.4f}")

**Resultado:** `TCP = 115,74 min` y `FI = 41,7 / 115,74 = 0,3603`. Un usuario está dispuesto a esperar el 36 % de su propio tiempo de carga (`PU = FI · TC`). El generador `generar_tc` respeta el rango observado y su media muestral cae dentro de la tolerancia de 4 errores estándar alrededor de `TCP`.

## 5. Obtención de las FDPs del Intervalo entre Arribos (IA)

El Intervalo de Arribos representa el tiempo entre que llega un vehículo al sistema y llega otro. Se utiliza la columna `Start Date` del dataset para calcular el tiempo entre llegadas, determinar la distribución estocástica que mejor represente este proceso, y poder generar valores aleatorios para dicha distribución.

**Supuestos de la serie de arribos.**
1. *Demanda que llega a cada estación.* El IA se calcula sobre los arribos de toda la red de Palo Alto y no de una estación aislada. En el modelo cada estación recibe el flujo de arribos completo y el filtro `PDCE` decide qué parte ingresa (propuesta), así que las FDPs representan la demanda total, antes de ese filtro. Junto con la sección 2.5: se supone que esa demanda es la de `T = 0` (1/7/2026), desde donde la escala el factor de crecimiento.
2. *Resolución de un minuto.* `Start Date` está registrado al minuto, y el 7,2 % de los arribos consecutivos cae en el mismo minuto que el anterior (IA nulo). El TP 4 reemplazaba los nulos por 0,5 min, lo que concentra probabilidad en ese valor y deja discreta el resto de la serie. Acá cada instante se distribuye uniformemente dentro de su minuto (se le suma un sorteo `U(0, 1)` min, con la semilla fija `SEMILLA_DATOS`), la corrección habitual para datos redondeados: elimina los nulos y la discretización sin cambiar la media.
3. *Franja de cada intervalo.* Cada IA se asigna a la franja y al tipo de día del arribo que lo abre, que es el instante en que el motor lo genera (`TPI(i) = T + IA`, con la FDP vigente en `T`). Un intervalo que cruza el borde de una franja queda en la franja de origen, igual que en el motor (propuesta).
4. *Tipo de día.* La franja nocturna (20:00–07:59) toma el tipo de día de la mañana en que termina: la del viernes es de fin de semana y la del domingo, de semana. Así, el fin de semana va del viernes a las 20:00 al domingo a las 19:59, y cada franja nocturna queda entera en un solo tipo de día, sin partirse a las 00:00. La noche del viernes precede a un día no laborable y la del domingo, a uno laborable. Los feriados no se distinguen (cuentan como días de semana). El reloj del motor (sección 9) tiene que usar la misma convención.

### 5.1. Segmentación por franja horaria y tipo de día

Franjas: 08:00–12:59, 13:00–19:59 y 20:00–07:59; días de semana y fin de semana.

In [ ]:
FRANJAS = {1: "08:00–12:59", 2: "13:00–19:59", 3: "20:00–07:59"}
SIGLAS_IA = ["IAS1", "IAS2", "IAS3", "IAF1", "IAF2", "IAF3"]

def franja_horaria(hora):
    """Franja de una hora del día (0–23): 1 = 08:00–12:59, 2 = 13:00–19:59, 3 = 20:00–07:59."""
    hora = np.asarray(hora)
    return np.select([(hora >= 8) & (hora < 13), (hora >= 13) & (hora < 20)], [1, 2], 3)

# Proporción de arribos consecutivos registrados en el mismo minuto (supuesto 2).
ia_registrado = df["Start Date"].diff().dt.total_seconds().dropna()
print(f"IA nulos con la resolución de un minuto: {100 * (ia_registrado == 0).mean():.1f} %")

# Instantes de arribo desagregados dentro de su minuto (supuesto 2), ordenados.
rng_datos = np.random.default_rng(SEMILLA_DATOS)
arribos = pd.DataFrame({"instante": df["Start Date"] + pd.to_timedelta(rng_datos.uniform(0, 60, len(df)), unit="s")})
arribos = arribos.sort_values("instante", ignore_index=True)
# IA que abre cada arribo: hasta el arribo siguiente de la red (supuestos 1 y 3); el último no abre ninguno.
arribos["IA"] = (arribos["instante"].shift(-1) - arribos["instante"]).dt.total_seconds() / 60
# Tipo de día (supuesto 4): fin de semana del viernes a las 20:00 al domingo a las 19:59. Correr el instante 4 h lleva
# las 20:00 a las 00:00 del día siguiente, así que la franja nocturna toma el día de la mañana en que termina.
fin_de_semana = (arribos["instante"] + pd.Timedelta(hours=4)).dt.day_of_week >= 5
arribos["sigla"] = "IA" + np.where(fin_de_semana, "F", "S") + franja_horaria(arribos["instante"].dt.hour).astype(str)
series_ia = {sigla: arribos.loc[arribos["sigla"] == sigla, "IA"].dropna().to_numpy() for sigla in SIGLAS_IA}

assert sum(len(serie) for serie in series_ia.values()) == len(df) - 1, "Algún intervalo quedó sin franja."
assert all((serie > 0).all() for serie in series_ia.values()), "Quedan intervalos nulos."
assert np.isclose(sum(serie.sum() for serie in series_ia.values()),
                  (arribos["instante"].iloc[-1] - arribos["instante"].iloc[0]).total_seconds() / 60), \
    "Los intervalos no cubren todo el período observado."

resumen_ia = pd.DataFrame([{"serie": sigla, "tipo de día": "semana" if sigla[2] == "S" else "fin de semana",
                            "franja": FRANJAS[int(sigla[3])], "intervalos": len(serie), "media (min)": serie.mean(),
                            "mediana (min)": np.median(serie), "percentil 99 (min)": np.quantile(serie, 0.99),
                            "máximo (min)": serie.max(), "arribos por hora": 60 / serie.mean()}
                           for sigla, serie in series_ia.items()]).set_index("serie")
display(resumen_ia.round(2))

fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
for ax, sigla in zip(ejes.flat, SIGLAS_IA):
    serie = series_ia[sigla]
    ax.hist(serie, bins=100, range=(0, np.quantile(serie, 0.99)), density=True, color=PALETA[0])
    ax.set_title(f"{sigla}: {resumen_ia.loc[sigla, 'tipo de día']}, {FRANJAS[int(sigla[3])]}", fontsize=11)
    ax.set_xlabel("Intervalo entre arribos (minutos, hasta el percentil 99)")
    ax.set_ylabel("Densidad de probabilidad")
    ax.grid(color="#e5e5e5", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 5.2. Ajuste en paralelo de las seis FDPs

In [ ]:
# Las seis series se ajustan a la vez, repartidas entre los procesos (serie × distribución).
ajustes_ia = ajustar_fdps(series_ia)
for sigla in SIGLAS_IA:
    mostrar_mejores_dist(ajustes_ia[sigla], sigla)

**Elección de las FDPs.** En las cuatro series diurnas la mejor por `ks_statistic` está también entre las mejores por `sumsquare_error`, con `ks_statistic` por debajo de 0,01. Las elegidas son `f` (F de Snedecor) y `ncf` (F no central), con cola de tipo Pareto: es la forma que toma una exponencial cuya tasa varía dentro de la franja, porque la demanda no es constante a lo largo de cinco o siete horas. Con posición y escala libres, `f` es la misma familia que `betaprime`; `ncf` le agrega un parámetro de no centralidad.
- **`IAS1` (semana, 08:00–12:59): `f`.** Primera por `ks_statistic` (0,0042) y por `sumsquare_error`, empatada con `betaprime`, que da el mismo ajuste. Por `kl_div` queda vigésima, pero con un valor (0,0039) del mismo orden que el de las mejores.
- **`IAS2` (semana, 13:00–19:59): `ncf`.** Primera por `ks_statistic` (0,0069) y por `kl_div`, y cuarta por `sumsquare_error`.
- **`IAS3` (semana, 20:00–07:59): `halfcauchy`.** Las cuatro primeras por `ks_statistic` quedan prácticamente empatadas, entre 0,0266 y 0,0268: `dpareto_lognorm`, `foldcauchy`, `halfcauchy` y `skewcauchy`. Truncadas al rango observado, las dos últimas, que dan el mismo ajuste, son las que menos subestiman la media (−21,8 %, contra −24,5 % y −23,4 %; ver la limitación de abajo). Se elige `halfcauchy`, tercera por `ks_statistic` (0,0267) y por `sumsquare_error`, que tiene sólo posición y escala.
- **`IAF1` (fin de semana, 08:00–12:59): `f`.** Primera por `ks_statistic` (0,0051) y cuarta por `sumsquare_error`.
- **`IAF2` (fin de semana, 13:00–19:59): `ncf`.** Primera por `ks_statistic` (0,0055) y segunda por `sumsquare_error`.
- **`IAF3` (fin de semana, 20:00–07:59): `fisk` (log-logística).** Primera por `ks_statistic` (0,0275) y cuarta por `sumsquare_error`. Es la serie más corta (2 498 intervalos) y la de peor ajuste.

**Limitación de las franjas nocturnas.** Las series `IAS3` e `IAF3` son bimodales: mezclan los intervalos cortos de la noche temprana con el hueco largo de la madrugada (de varias horas, hasta 12, entre el último arribo de la noche y el primero de la mañana). Ninguna de las 108 distribuciones reproduce las dos modas a la vez, y truncadas al rango observado subestiman la media: las 20 mejores por `ks_statistic` de cada serie lo hacen en un 16 % a 27 %. Con las elegidas, la media del IA nocturno es un 22 % (semana) y un 25 % (fin de semana) menor que la observada, así que el modelo sobreestima la demanda nocturna en esa proporción. Como la franja nocturna concentra la menor parte de los arribos (1,4 y 0,9 por hora, contra entre 6,8 y 11,4 en las diurnas), el efecto sobre los resultados agregados es acotado, pero se tiene en cuenta al leer los resultados de esa franja.

In [ ]:
ELECCION_IA = {"IAS1": "f", "IAS2": "ncf", "IAS3": "halfcauchy", "IAF1": "f", "IAF2": "ncf", "IAF3": "fisk"}

FDP_IA = {sigla: obtener_dist(ajustes_ia[sigla], funcion, sigla, series_ia[sigla]) for sigla, funcion in ELECCION_IA.items()}

# El IA de cada arribo lo genera el motor (sección 9) con la FDP de la franja y el tipo de día vigentes, multiplicada
# por el factor de crecimiento: FDP_IA[sigla].rvs_truncado(rng) * factor_crecimiento(T).
rng_chequeo = np.random.default_rng(SEMILLA_DATOS)
verificacion_ia = pd.DataFrame([verificar_generador(FDP_IA[sigla], FDP_IA[sigla].rvs_truncado, series_ia[sigla], sigla,
                                                    rng_chequeo) for sigla in SIGLAS_IA]).set_index("serie")
display(verificacion_ia.round(3))

Los seis generadores respetan el rango observado y su media muestral reproduce la de su FDP truncada dentro de la tolerancia. La media de la FDP truncada difiere de la observada en menos de 1,6 % en las cuatro series diurnas. En las nocturnas la diferencia es de −21,8 % (`IAS3`) y −25,0 % (`IAF3`): es la limitación señalada arriba.

## 6. Obtención de las FDPs de Falla (TFC) y Reparación (TRC) de Cargadores

`TFC` es el tiempo desde la puesta en servicio (reparación) de un cargador hasta su falla siguiente; `TRC`, el tiempo que permanece fuera de servicio hasta su reparación.

**Fuente.** Los reportes de fallas depurados en 3.2 (`fallas`).

**Supuestos adoptados:**
1. *Transferibilidad.* No hay un registro público de fallas de cargadores de CABA. Se supone que la confiabilidad de los cargadores de CABA es la de la red escocesa.
3. *Reparación = cierre del reporte.* El cierre registra la puesta en servicio. Un cierre administrativo posterior a la reparación real alarga `TRC`.
3. *Episodios de falla.* Un cargador puede acumular varios reportes abiertos a la vez: el mismo corte reportado más de una vez. Los reportes de un cargador que se superponen se agrupan en un **episodio**, que empieza con el primer reporte y termina con el último cierre. `TRC` es la duración de cada episodio, y `TFC`, el tiempo entre el cierre de un episodio y el primer reporte del siguiente del mismo cargador, que es la definición de la propuesta (de la reparación a la falla siguiente).
4. *Censura.* Los reportes que siguen abiertos al final del registro no tienen `TRC` ni abren un `TFC`. Además, `TFC` sólo se observa en cargadores con al menos dos episodios dentro de los 13 meses del registro, y no puede superar esa ventana: los cargadores más confiables quedan subrepresentados y `TFC` tiende a subestimarse. El sesgo es conservador, porque el modelo tendrá más fallas que la red real.

In [ ]:
def episodios_de_falla(fallas):
    """Agrupa los reportes superpuestos de cada cargador en episodios de falla (supuesto 3). Un reporte anterior al fin
    del episodio en curso se suma a él y extiende su fin; un reporte sin cierre deja el episodio abierto (fin = NaT), y
    el reporte siguiente del mismo cargador abre otro episodio, sin TFC entre los dos. Espera los reportes ordenados por
    cargador y por instante."""
    filas = []
    for cargador, grupo in fallas.groupby("cargador", sort=False):
        inicio = fin = None
        for reporte, cierre in zip(grupo["reporte"], grupo["cierre"]):
            if inicio is not None and pd.notna(fin) and reporte <= fin:
                fin = max(fin, cierre) if pd.notna(cierre) else pd.NaT
                cantidad += 1
            else:
                if inicio is not None:
                    filas.append((cargador, inicio, fin, cantidad))
                inicio, fin, cantidad = reporte, cierre, 1
        filas.append((cargador, inicio, fin, cantidad))
    return pd.DataFrame(filas, columns=["cargador", "inicio", "fin", "reportes"])

episodios = episodios_de_falla(fallas)
episodios["TRC"] = (episodios["fin"] - episodios["inicio"]).dt.total_seconds() / 60
episodios["TFC"] = (episodios.groupby("cargador")["inicio"].shift(-1) - episodios["fin"]).dt.total_seconds() / 60
series_falla = {"TFC": episodios["TFC"].dropna().to_numpy(), "TRC": episodios["TRC"].dropna().to_numpy()}

# Chequeos: cada reporte cae en un solo episodio, y los episodios de un cargador no se superponen ni se tocan.
assert episodios["reportes"].sum() == len(fallas), "Hay reportes fuera de los episodios o contados dos veces."
assert (series_falla["TRC"] > 0).all() and (series_falla["TFC"] > 0).all(), "Hay episodios superpuestos o vacíos."
print(f"{len(episodios):,} episodios de falla ({(episodios['reportes'] > 1).sum():,} agrupan más de un reporte; "
      f"{episodios['fin'].isna().sum()} abiertos). Cargadores con al menos dos episodios: "
      f"{(episodios.groupby('cargador').size() > 1).sum():,}.")

MIN_POR_DIA = 24 * 60
resumen_falla = pd.DataFrame([{"serie": nombre, "datos": len(serie), "media (días)": serie.mean() / MIN_POR_DIA,
                               "mediana (días)": np.median(serie) / MIN_POR_DIA,
                               "percentil 5 (min)": np.quantile(serie, 0.05),
                               "percentil 95 (días)": np.quantile(serie, 0.95) / MIN_POR_DIA,
                               "máximo (días)": serie.max() / MIN_POR_DIA}
                              for nombre, serie in series_falla.items()]).set_index("serie")
display(resumen_falla.round(2))

# Escala logarítmica: los dos tiempos van de minutos a meses.
fig, ejes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (nombre, serie), titulo in zip(ejes, series_falla.items(),
                                       ["Tiempo hasta la falla (TFC)", "Tiempo de reparación (TRC)"]):
    ax.hist(serie, bins=np.logspace(np.log10(serie.min()), np.log10(serie.max()), 60), color=PALETA[0])
    ax.set_xscale("log")
    ax.set_title(f"{titulo}: {len(serie):,} datos", fontsize=11)
    ax.set_xlabel("Minutos (escala logarítmica)")
    ax.set_ylabel("Cantidad de episodios")
    ax.grid(color="#e5e5e5", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# TFC y TRC se ajustan a la vez, repartidas entre los procesos (serie × distribución).
ajustes_falla = ajustar_fdps(series_falla)
for nombre in series_falla:
    mostrar_mejores_dist(ajustes_falla[nombre], nombre)

**Elección de las FDPs.**
- **`TFC`: `exponweib` (Weibull exponenciada).** Primera por `ks_statistic` (0,0174), sexta por `sumsquare_error` y por `kl_div`, y su media truncada coincide con la observada (40,3 días). Sus dos parámetros de forma son menores que 1, lo que da una tasa de falla decreciente desde la puesta en servicio: un cargador recién reparado tiene más probabilidad de volver a fallar pronto, coherente con las reparaciones que no resuelven la causa de fondo.
- **`TRC`: `erlang` con forma no entera, que es la gamma.** Ninguna distribución ajusta bien: el mejor `ks_statistic` es 0,0669. La serie mezcla dos procesos, los reinicios remotos que cierran el reporte en minutos (el 5 % de los episodios dura menos de un minuto) y las reparaciones en el lugar, que llevan días o semanas, y ninguna distribución unimodal reproduce las dos modas. La primera por `ks_statistic` es `erlang`, con un parámetro de forma de 0,295. La Erlang es la gamma de forma entera, pero `scipy` no impone esa restricción al ajustarla, así que el resultado es una gamma de forma 0,295; la `gamma` de `scipy`, en cambio, no converge desde su punto de partida y queda con un `ks_statistic` de 0,999. Con forma menor que 1, la densidad decrece desde cero: concentra probabilidad en las reparaciones cortas y deja una cola larga para las de días o semanas. Por `sumsquare_error` y `kl_div` queda trigésima tercera y trigésima, pero su media truncada difiere en −0,2 % de la observada. Las que la siguen por `ks_statistic`, `ncf` y `f`, subestiman la media en 10,7 % y 9,5 %, y `gengamma`, séptima (0,1075), la sobreestima en 5,8 %. Como `TRC` define cuánto tiempo pasa un cargador fuera de servicio, la media es la propiedad que más pesa en los resultados (`PDC`, `PAPF`): se elige `erlang`. El mismo criterio usó el TP 4 al elegir `landau` para el IA.

In [ ]:
FDP_TFC = obtener_dist(ajustes_falla["TFC"], "exponweib", "TFC", series_falla["TFC"])
# erlang con forma no entera es la gamma (ver la elección): scipy lo avisa en cada llamada, sin afectar los valores.
warnings.filterwarnings("ignore", message="The shape parameter of the erlang distribution has been given a non-integer value")
FDP_TRC = obtener_dist(ajustes_falla["TRC"], "erlang", "TRC", series_falla["TRC"])

def generar_tfc(rng=np.random):
    """Generador estocástico del Tiempo hasta el Fallo de un Cargador, desde su puesta en servicio."""
    return FDP_TFC.rvs_truncado(rng)

def generar_trc(rng=np.random):
    """Generador estocástico del Tiempo para Reparar un Cargador."""
    return FDP_TRC.rvs_truncado(rng)

rng_chequeo = np.random.default_rng(SEMILLA_DATOS)
verificacion_falla = pd.DataFrame([verificar_generador(FDP_TFC, generar_tfc, series_falla["TFC"], "TFC", rng_chequeo),
                                   verificar_generador(FDP_TRC, generar_trc, series_falla["TRC"], "TRC", rng_chequeo)]
                                  ).set_index("serie")
display(verificacion_falla.round(1))
# Disponibilidad de largo plazo de un cargador sin mantenimiento preventivo: E[TFC] / (E[TFC] + E[TRC]).
print(f"Disponibilidad implícita sin mantenimiento preventivo: {FDP_TFC.media() / (FDP_TFC.media() + FDP_TRC.media()):.1%}")

Los dos generadores respetan el rango observado y reproducen la media de su FDP truncada dentro de la tolerancia. Con las FDPs elegidas, un cargador sin mantenimiento preventivo pasa en promedio 40,3 días en servicio entre fallas y 14,1 días fuera de servicio, una disponibilidad de largo plazo de 74,1 %. Es la referencia contra la que se mide el efecto del mantenimiento preventivo (`TMP`), que repara los cargadores fallados sin esperar a que termine su `TRC`.

## 7. Obtención del Costo Energético en función del TC

El Costo Energético representa el dinero gastado en la energía de los cargadores. Este depende de:
- El Tiempo de Carga (TC)
- El Precio de la Energía (PE): Valor constante que representa cuanto vale el kWh en CABA en Pesos

Para calcularlo, usaremos regresión lineal para obtener una función que nos permita pasar de TC a CE.


> **[DESACTUALIZADO — TP 4]** La regresión del TP 4 ordena por separado tiempo y energía (`np.sort`), lo que no es una regresión sobre los pares reales e infla el R². Rehacerla sobre los pares `(TC_k, kWh_k)` alineados y reportar el R² verdadero. El precio único `PE` se reemplaza por tarifas por franja (pico/valle). Además, la sigla `CE` del TP 4 (costo energético) choca con `CE` de la propuesta (cantidad de estaciones): renombrar.

In [ ]:
# [DESACTUALIZADO — TP 4] regresión sobre series ordenadas por separado; rehacer sobre pares alineados.
PE = 108.48 # Precio Promedio de Energia en CABA, en Pesos

tiempo = df["Charging Time (minutes)"].values
energia = df['Energy (kWh)'].dropna().values

slope, intercept, r, p, se = stats.linregress(
    np.sort(tiempo),
    np.sort(energia)
)

print(f"Energía [kWh] = {slope:.4f} x Tiempo [min] + {intercept:.4f}  (R²={r**2:.4f})")
print(f"Costo [ARS] = Energía [kWH] x {PE} [ARS/kWH]")

def tiempo_a_costo_lineal(t):
    return float((slope * t + intercept) * PE)

### 7.1. Tarifas de energía por franja horaria

In [ ]:
# [PENDIENTE] Precio de la energía por franja (pico/valle) con su fuente o supuesto explicitado.

## 8. Constantes y Parámetros del Modelo

Constantes según la propuesta: `CC_MAX`, `CE_MAX`, `CCP`, `CPN`, `CEN`, `TIC`, `TCE`, `PUNEN`, `FI`, `TCP`; y valores por defecto de las variables de control `RC`, `TMP` y `PDCE`.

> **[PENDIENTE]** Faltan los valores de `CC_MAX`, `CE_MAX`, `CPN`, `CEN`, `CCP` y los costos de mantenimiento. Cada supuesto se explicita con su justificación.

In [ ]:
# [PENDIENTE] Definición de constantes y valores por defecto de las variables de control.

## 9. Funciones Auxiliares

Reloj (mapeo de `T` a hora del día y tipo de día), generación del `IA` con la FDP vigente y el factor de crecimiento, prorrateo de acumuladores y costo de energía entre franjas, y decisión de arrepentimiento (`PUNEN`, `PU`, `EEU`).

In [ ]:
# [PENDIENTE] Funciones auxiliares del motor (a nivel de módulo, para que sean serializables).

def franja_y_tipo_dia(T):
    raise NotImplementedError("Pendiente: mapeo de T a franja y tipo de día (sección 9).")

def generar_ia(T):
    raise NotImplementedError("Pendiente: IA con la FDP vigente y el factor de crecimiento (sección 9).")

## 10. Implementación del Motor de Simulación

Se define un modelo de simulación de eventos discretos para un sistema multi-servidor (N y N) con lógica de abandono (arrepentimiento). El sistema gestiona una lista de eventos futuros basada en arribos y salidas por cada puesto.

> **[DESACTUALIZADO — TP 4]** Motor del TP 4: un solo IA global, elección aleatoria de puesto, arrepentimiento hardcodeado y estado en variables globales. Queda como referencia de la estructura del loop; se reemplaza por el motor de las subsecciones 10.1–10.3.

In [ ]:
# [DESACTUALIZADO — TP 4] motor de referencia; no usar para el TP Final.
# Variables de Control
N: int                        # Cantidad de puestos

#  Estado del reloj y eventos
T: float                      # Reloj actual
TF: float                     # Horizonte de simulación
TPLL: float                   # Próxima llegada global
TPS: List[float]              # Próxima salida por puesto

#  Estado del sistema
NS: List[int]                 # Autos en cada puesto (cola + servicio)

#  Contadores y acumuladores
CLL: int                      # Llegadas efectivas
CARR: int                     # Arrepentimientos
SPS: float                    # suma ponderada de personas en sistema
STC: float                    # Suma de tiempos de carga

ITO: List[float]              # Instante último cambio a ocioso por puesto
STO: List[float]              # Tiempo ocioso acumulado por puesto
SCE: List[float]              # Costo Energético acumulado por puesto

#  Resultados
PPS: float                    # Promedio de personas en sistema
PEC: float                    # Promedio en cola
PARR: float                   # % arrepentimiento
PTO: List[float]              # Proporción de ocio por puesto
CCE: List[float]              # Costo de Consumo Energetico por puesto

# Factor de comportamiento: probabilidad de que un usuario abandone la cola
PORCENTAJE_ARREPENTIMIENTO = 0.93

def condiciones_iniciales(cant_puestos: int):
    global TF, N
    global T, TPLL, TPS
    global NS
    global CLL, CARR, SPS, STC, ITO, STO, SCE

    # Variables de Control
    N = int(cant_puestos)

    # Reloj/eventos
    T = 0.0
    TF = 1000000.0
    TPLL = 0.0

    # Acumuladores
    CLL = 0
    CARR = 0
    SPS = 0.0
    STC = 0.0

    # Vectores
    TPS = [float("inf")] * N
    NS = [0] * N
    ITO = [0.0] * N
    STO = [0.0] * N
    SCE = [0.0] * N

def arrepentimiento(index_puesto) -> bool:
    global CARR, NS, PORCENTAJE_ARREPENTIMIENTO
    cant_autos = NS[index_puesto]

    if cant_autos <= 1:
        return False
    elif cant_autos <= 2:
        r = random.random()
        if r >= PORCENTAJE_ARREPENTIMIENTO:
            return False
        else:
            CARR += 1
            return True
    else:
        CARR += 1
        return True

def llegada():
    global TPLL, SPS, T, NS, CLL, STC, STO, ITO, TPS, N

    SPS += (TPLL - T) * (sum(NS))

    T = TPLL #Avanzamos en el tiempo

    IA = generar_ia() #Generamos el tiempo hasta la proxima llegada
    TPLL = T + IA #Calculamos el tiempo de llegada del proximo auto

    index_puesto = eleccion_server() #Subrutina
    if arrepentimiento(index_puesto):
        return

    # Llegada efectiva
    NS[index_puesto] += 1
    CLL += 1

    if NS[index_puesto] == 1:
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC
        STO[index_puesto] += (T - ITO[index_puesto])



def salida(index_puesto):
    global SPS, T, NS, STC, ITO, STO, TPS

    SPS += (TPS[index_puesto] - T) * (sum(NS))

    T = TPS[index_puesto]  # Avanzamos en el tiempo

    NS[index_puesto] -= 1  # Actualizo vector estado

    if NS[index_puesto] > 0:  # Evento Futuro Condicionado
        TC = generar_tc()
        STC += TC
        SCE[index_puesto] += tiempo_a_costo_lineal(TC)
        TPS[index_puesto] = T + TC  # Actualizo vector de tiempos de salida
    else:
        ITO[index_puesto] = T
        TPS[index_puesto] = float("inf")  # Actualizo vector de tiempos de salida

def eleccion_server():
    global N
    return random.randrange(N) # Distribución uniforme de la demanda

def menor_tps_index():
    global TPS
    return min(range(len(TPS)), key=TPS.__getitem__)

def calculo_resultados():
    global SPS, STC, SCE, CLL, CARR, STO, T, N
    total_arribos = CLL + CARR

    PPS = SPS / CLL if CLL > 0 else 0.0
    PEC = (SPS - STC) / CLL if CLL > 0 else 0.0
    PARR = CARR * 100 / total_arribos if total_arribos > 0 else 0.0

    PTO = []
    CCE = []

    for i in range(N):
        PTO.append(STO[i] * 100 / T if T > 0 else 0.0)

    for i in range(N):
        CCE.append(SCE[i] * 100 / CLL if CLL > 0 else 0.0)

    print(f"PPS = {round(PPS,2)}")
    print(f"PEC = {round(PEC,2)}")
    print(f"PTO = {[round(x, 2) for x in PTO]}")
    print(f"CCE = {[round(x, 2) for x in CCE]}")
    print(f"PARR = {round(PARR,2)}")


def simular(cant_puestos: int):
    global TPLL, TPS, T, TF, NS

    condiciones_iniciales(cant_puestos)

    while True:
        index_server = menor_tps_index()

        if TPLL <= TPS[index_server]:
            llegada()
        else:
            salida(index_server)

        if T <= TF:
            continue
        else: # Vaciamiento
            if any(x > 0 for x in NS):
                TPLL = float("inf")
                continue
            break
    calculo_resultados()

### 10.1. Estado de la simulación

In [ ]:
# [PENDIENTE] Estado encapsulado (dataclass): variables de estado CA(i), CD(i), CC(i), CE;
# TEF TPI(i), TPC(i)(j), TPAE, TPIC(i), TPCE, TPFC(i)(j), TPRC(i)(j), TPMP(i);
# auxiliares CARRUM(i), CPAACUM, CAPF(i), ICC(i)(j), CTC(i)(j) y acumuladores por franja.

### 10.2. Rutinas de eventos

Una rutina por evento de la tabla de eventos y condiciones de la propuesta.

In [ ]:
# [PENDIENTE] Rutinas: Ingreso de auto a estación (i), Carga en cargador (i)(j),
# Análisis de Expansión, Instalación de nuevo cargador (i), Construcción de nueva estación,
# Falla de cargador (i)(j), Reparación de cargador (i)(j), Mantenimiento preventivo (i).

### 10.3. Función de simulación

Función pura: recibe parámetros y semilla, devuelve las métricas de resultado (`BM`, `PTO`, `PEC`, `PPS`, `PARR`, `PDC`, `PAPF` por franja y sus promedios).

In [ ]:
# [PENDIENTE] Loop evento a evento (avance al mínimo de la TEF) y cálculo de resultados.

def simular_tp_final(RC, TMP, PDCE, TF, semilla):
    raise NotImplementedError("Pendiente: motor del TP Final (sección 10.3).")

## 11. Verificación y Validación del Motor

Invariantes y aserciones derivados de la tabla de eventos y condiciones de la propuesta, y validación contra el modelo analítico M/M/c en un escenario reducido.

In [ ]:
# [PENDIENTE] Chequeos sobre valores y cambios de estado (no sobre "no tiró excepción"),
# p. ej. autos cargando = min(CA(i), CD(i)), y comparación contra M/M/c.

## 12. Experimentación y Análisis de Sensibilidad

Se ejecutan múltiples escenarios variando el número de puestos de carga (N) para identificar el punto de equilibrio operativo donde se minimiza el arrepentimiento y el tiempo de espera en cola, manteniendo una utilización eficiente del cargador.

> **[DESACTUALIZADO — TP 4]** El barrido del TP 4 varía la cantidad de puestos `N`. En el TP Final la cantidad de cargadores y estaciones es dinámica; los escenarios varían `RC`, `TMP` y `PDCE`.

In [ ]:
# [DESACTUALIZADO — TP 4] barrido sobre N; reemplazar por la grilla de RC, TMP y PDCE.
# Simulación para configuraciones de 1 a 40 puestos
for i in range(1, 41):
    print(f"\n--- ESCENARIO: N = {i} puestos ---")
    simular(cant_puestos=i)

### 12.1. Grilla de escenarios y réplicas en paralelo

In [ ]:
# [PENDIENTE] Grilla de combinaciones (RC, TMP, PDCE) × réplicas, ejecutada con
# ProcessPoolExecutor (workers según os.cpu_count()), con semilla por corrida derivada
# de una semilla base + índice de combinación/réplica.

### 12.2. Análisis de resultados

In [ ]:
# [PENDIENTE] Tablas y gráficos comparativos de las métricas por escenario, con intervalos de confianza.

## 13. Conclusiones

*[PENDIENTE]*